# 🌿 Korea's Sector-Level Carbon Footprint Analysis -- the KR-EEIO Model

---

This notebook combines the **Bank of Korea Input-Output Table (IO Table)** with
**sector-level greenhouse gas emissions data** to quantitatively analyze **how much carbon
each sector induces across the whole supply chain** when it produces KRW 1 million worth of output.

| Basis | Use |
|-----------|------|
| **Current prices** (nominal, that year's market prices) | Company-level simulator use, carbon calculations on an absolute-amount basis |
| **Constant prices** (relative prices, 2020 base year)   | Year-over-year **time-series comparison**, analysis with price effects removed |

> ⚙️ **This notebook is self-contained.** The calculation and plotting code from
> `eeio_core.py` is copied directly into the setup cell below, so this notebook computes
> everything and generates every image/graph itself -- it does not `import eeio_core`.

---

All code and data for the KR-EEIO carbon-emissions analysis pipeline are available on the
[Data Science Team/kr_eeio Gitlab](https://bidas-gitlab.boknet.intra/2620316/kr_eeio/-/tree/main/) page.


In [ ]:
import os
import textwrap

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from IPython.display import display, HTML

# -- Korean font setup --------------------------------------------------------------
import matplotlib.font_manager as fm

_candidates = [
    f.name for f in fm.fontManager.ttflist
    if any(k in f.name for k in ["Nanum", "Malgun", "AppleGothic", "Gulim", "NanumGothic"])
]
_font = _candidates[0] if _candidates else "DejaVu Sans"
plt.rcParams["font.family"] = _font
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"

# Shared M-matrix store (by year)
M_matrices: dict = {}

# ═══════════════════════════════════════════════════════════════════════════════
# Common utility functions (from eeio_core.py)
# ═══════════════════════════════════════════════════════════════════════════════

def _find_block(df: pd.DataFrame, code_col: int = 0):
    """Dynamically finds the start/end rows of the sector block in the code column (A..T)."""
    codes = df.iloc[:, code_col].astype(str).str.strip().values
    start = int(np.where(codes == "A")[0][0])
    end   = int(np.where(codes == "T")[0][0])
    return start, end, codes


def _pick_sheet(xls: pd.ExcelFile, kw: str, p_type: str) -> str:
    """Select a sheet name by keyword + price type (handles underscore/space naming variants)."""
    m = [s for s in xls.sheet_names if kw in s and p_type in s]
    if not m:
        m = [s for s in xls.sheet_names if kw in s]
    if not m:
        raise ValueError(f"'{kw}'({p_type}) the sheet could not be found.")
    return m[0]


def _total_input_vector(df_tot: pd.DataFrame, size: int, code_col: int = 0) -> np.ndarray:
    """Total_Transaction sheet: the 'Total input' (9790) row -> a total-input vector, by consuming sector, per column."""
    rc = df_tot.iloc[:, code_col].astype(str).str.strip().values
    cand = np.where(rc == "9790")[0]
    if not len(cand):
        lbl  = df_tot.iloc[:, 1].astype(str).str.strip().values
        cand = np.where(lbl == "Total input")[0]
    if not len(cand):
        raise ValueError("Total_Transaction in 'Total input'(9790) the row could not be found.")
    return df_tot.iloc[int(cand[0]), 2:2 + size].values.astype(float)


def clean_industry_names(idx: pd.Index) -> pd.Index:
    """Normalizes whitespace in the sector-name index.

    Note: since the translated sector names are all looked up from a single
    authoritative source (the classification file's code -> English-name
    mapping), the spacing inconsistencies that the original Korean source
    data had across years (e.g. "SocialWelfareServices" vs "Social Welfare
    services") no longer occur here. This is kept as a light strip/collapse
    for defensive purposes only.
    """
    return (
        idx.astype(str).str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


def clean_and_merge_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize sector names in the pivot table and merge duplicate rows."""
    df = df.copy()
    df.index = clean_industry_names(df.index)
    return df.groupby(df.index).max()


def out_path(filename: str, output_dir: str = "./output") -> str:
    """Helper function that returns a path under output_dir."""
    os.makedirs(output_dir, exist_ok=True)
    return os.path.join(output_dir, filename)

# ═══════════════════════════════════════════════════════════════════════════════
# Data loading
# ═══════════════════════════════════════════════════════════════════════════════

def load_ghg_data(
    ghg_file: str = "O_2015-2023_Industry_GHG_Estimates_FF_EN.xlsx",
) -> pd.DataFrame:
    """
    Loads the per-sector greenhouse gas emissions data (unit: kt CO2eq).

    Reads with header=1, the same as the original master builder, to keep
    the 'Code' column.
    - Index: numeric (reset)
    - 'Code' column: sector code (A, B, C01, ...)
    - 'Sector name' column: sector name
    - Year columns: '2015', '2016', ... (str)

    Returns
    -------
    df : pd.DataFrame  (includes the 'Code' column; excludes code T)
    """
    df = pd.read_excel(ghg_file, sheet_name="Annual_Summary_Final", header=1)

    # column name cleanup: float → int → str (e.g. 2015.0 → '2015')
    new_cols = []
    for c in df.columns:
        try:
            new_cols.append(str(int(float(str(c)))))
        except (ValueError, TypeError):
            new_cols.append(str(c).strip())
    df.columns = new_cols

    # code column cleanup
    df = df[df['Code'].notna()].copy()
    df['Code'] = df['Code'].astype(str).str.strip()

    # Sector name column unify ('Sector name' → 'Sector name')
    if 'Sector name' in df.columns and 'Sector name' not in df.columns:
        df = df.rename(columns={'Sector name': 'Sector name'})

    # sector code: keep only rows starting with a letter (excludes aggregate/final-demand rows like 9090, 9111)
    # and among those, exclude the row that is exactly 'T' (Others)
    df = df[df['Code'].str.match(r'^[A-Za-z]', na=False)].copy()
    df = df[df['Code'] != 'T'].reset_index(drop=True)

    year_cols = [c for c in df.columns if c.isdigit() and 2000 <= int(c) <= 2100]
    print(f"✅ GHG Data loaded  |  number of sectors: {len(df)}  |  Year: {year_cols}")
    return df

# ═══════════════════════════════════════════════════════════════════════════════
# EEIO calculation (shared by Current/Constant prices)
# ═══════════════════════════════════════════════════════════════════════════════

def run_eeio(
    year,
    file_name: str,
    ghg_df: pd.DataFrame,
    p_type: str = "Current",
    exclude_keyword: str | None = None,
    ghg_unit_scale: float = 1000.0,
) -> pd.DataFrame | None:
    """
    National domestic technical-coefficient-matrix based EEIO coefficients
    (direct / supply-chain / Scope1/2/3) calculation.

    Parameters
    ----------
    year             : analysis year (int or str)
    file_name        : path to that year's IO Excel file
    ghg_df           : return value of load_ghg_data() (kt CO2eq)
    p_type           : 'Current' or 'Constant'
    exclude_keyword  : sector-name prefix to exclude (e.g. 'Others')
                       * excludes only rows that start exactly with 'Others'
                         (does NOT include e.g. 'Other services')
    ghg_unit_scale   : kt -> t conversion factor (default 1000)

    Returns
    -------
    df_result : pd.DataFrame, index = Sector name
        Columns: Direct emission coefficient(B), Supply chain induced coefficient(B*L), Indirect induced amount(BL-B), Scope2, Scope3
    """
    try:
        xls = pd.ExcelFile(file_name)

        tot_sheet = _pick_sheet(xls, "Total_Transaction",  p_type)
        dom_sheet = _pick_sheet(xls, "Domestic_Transaction", p_type)
        df_tot = pd.read_excel(file_name, sheet_name=tot_sheet, header=None)
        df_dom = pd.read_excel(file_name, sheet_name=dom_sheet, header=None)

        s_t, e_t, codes_t = _find_block(df_tot)
        s_d, e_d, codes_d = _find_block(df_dom)
        sector_codes   = codes_d[s_d:e_d + 1]
        if not np.array_equal(codes_t[s_t:e_t + 1], sector_codes):
            raise ValueError("The Total Transaction and Domestic Transaction tables have different sector-code orders.")

        # * [FIX] Normalize the sector names right here (the existing bug: the
        #   industry_names created below used to be stored raw, as-is, in
        #   M_matrices/Label, while the dashboard referred to the normalized
        #   (clean) name -- the notation differed, causing the match to fail
        #   (e.g. 2015 "SocialWelfareServices" vs "Social Welfare services").
        #   Normalizing right at the source like this means M_matrices['Sector name']/['Label']
        #   and df_result's index always use the same notation, guaranteeing consistency.
        #   The existing clean_industry_names() call on df_result
        #   is now just re-normalizing an already-normalized value, so it is harmless (idempotent).
        industry_names = clean_industry_names(
            pd.Index(df_dom.iloc[s_d:e_d + 1, 1].values.astype(str))
        ).values
        size = len(sector_codes)

        idx_D = np.where(sector_codes == "D")[0]
        if not len(idx_D):
            raise ValueError("No electricity (D) code found.")
        D_rel = int(idx_D[0])

        X  = _total_input_vector(df_tot, size)
        Xs = np.where((X == 0) | np.isnan(X), np.nan, X)

        Z = df_dom.iloc[s_d:e_d + 1, 2:2 + size].values.astype(float)
        A = np.nan_to_num(Z / Xs.reshape(1, -1), nan=0.0, posinf=0.0, neginf=0.0)
        L = np.linalg.inv(np.eye(size) - A)

        # GHG: set the 'Code' column as the index, then extract the Year column
        # * Important: the kt -> t conversion (ghg_unit_scale, default factor of 1000) is applied
        #   before the B/M/M_X calculation. The previous version applied this conversion only at the
        #   df_result display step, leaving a bug where the M/B/BL/M_X stored in M_matrices
        #   stayed in kt units (i.e. the actual values used in the dashboard/Excel were 1000x too small).
        _yr_key = str(year)
        if _yr_key not in ghg_df.columns:
            raise KeyError(f"GHG data has no column for year {year}. (available: {list(ghg_df.columns)})")
        E_kt = (
            ghg_df.set_index('Code')[_yr_key]
            .reindex(pd.Index(sector_codes.tolist()))
            .fillna(0)
            .values
            .astype(float)
        )
        E = E_kt * ghg_unit_scale   # kt -> t conversion (here applied right away)
        B  = np.nan_to_num(E / Xs, nan=0.0, posinf=0.0, neginf=0.0)
        BL = (B.reshape(1, -1) @ L).flatten()
        M  = np.diag(B) @ L

        # ── M_X: value-induced matrix (M x diag(X)) ──────────────────────────────
        # M[i,j] = the emission coefficient induced via sector i when sector j produces KRW 1 million
        # (per-unit basis, t CO2eq / KRW million). Multiplying this by sector j's actual total
        # input value X[j] gives the absolute emissions (t CO2eq) actually induced via sector i
        # at sector j's actual production scale. This is unified so that the dashboard cards'
        # Upstream/Downstream table and the Excel M_X sheet reference the same values.
        X_for_scale = np.nan_to_num(X, nan=0.0)
        M_X = M @ np.diag(X_for_scale)

        # -- Code+name MultiIndex Label (Row: code row + name row, Column: code column + name column,
        #    an array of (Code, name) tuples for showing them split apart) ----------------------
        labels = pd.MultiIndex.from_arrays(
            [sector_codes.tolist(), industry_names.tolist()],
            names=["Code", "Sector name"],
        )

        # * The key includes not just the Year but also p_type (Current/Constant). Previously the key
        #   was the year alone, so computing the same year repeatedly in Current->Constant (or the
        #   reverse) order would overwrite the earlier result with the later one, and then
        #   build_eeio_matrices() would say "already computed, so reuse it" and build the Excel file
        #   with the wrong price-basis value -- this was the root cause of a bug where the dashboard
        #   Upstream/Downstream and the Excel M.X(diag) values ended up differing).
        mkey = f"{year}_{p_type}"
        M_matrices[mkey] = {
            "M": M.copy(), "M_X": M_X.copy(), "L": L.copy(), "A": A.copy(),
            "B": B.copy(), "BL": BL.copy(), "X": X.copy(), "GHG_raw": E.copy(),
            "Sector name": industry_names.copy(), "Code": sector_codes.copy(),
            "Label": labels,
        }
        # For backward compatibility: if code elsewhere still accesses the cache by "year only" as the key,
        # also update the year-only key so it keeps showing the latest calculation result (for reference).
        M_matrices[year] = M_matrices[mkey]

        # Scope 1 / 2 / 3 (already in t units B/M/BL based on calculate — additional scale not needed)
        scope_1     = B.copy()
        scope_2_raw = M[D_rel, :].copy()
        scope_2_raw[D_rel] -= scope_1[D_rel]
        scope_2 = np.maximum(scope_2_raw, 0.0)

        scope_3_raw = BL - scope_1 - scope_2
        scope_3 = np.maximum(scope_3_raw, 0.0)

        neg_n = int((scope_3_raw < -1e-9).sum())
        if neg_n:
            print(f"⚠️ {year}: Scope3 negative {neg_n} (clipped and lost). "
                  f"min={scope_3_raw.min():.4g} — recommend reviewing the definition")

        df_result = pd.DataFrame({
            "Sector name":             industry_names,
            "Direct emission coefficient (B)":     scope_1,
            "Supply chain induced coefficient (B*L)": BL,
            "Indirect induced amount (BL-B)":   (BL - scope_1),
            "Scope2":            scope_2,
            "Scope3":            scope_3,
        })

        # -- exclude only sectors that start with exactly 'Others' -----------------------
        # str.startswith('Others') would also match 'Other services', 'Other finance', etc.,
        # so we exclude only rows where the sector name is exactly 'Others' or starts with 'Others '.
        if exclude_keyword:
            mask = df_result["Sector name"].astype(str).str.strip() == exclude_keyword
            df_result = df_result[~mask].copy()

        df_result["Sector name"] = clean_industry_names(pd.Index(df_result["Sector name"].astype(str)))
        return df_result.set_index("Sector name")

    except Exception as e:
        print(f"❌ {year}: an error occurred: {e}")
        return None

# ═══════════════════════════════════════════════════════════════════════════════
# Full-year batch run
# ═══════════════════════════════════════════════════════════════════════════════

def run_all_years(
    io_files: dict,
    analysis_years: list,
    ghg_df: pd.DataFrame,
    p_type: str = "Current",
    exclude_keyword: str | None = None,
    output_dir: str = "./output",
) -> dict:
    """Full-year EEIO analysis run → final_results dict returns."""
    final_results = {}
    label = "Current" if p_type == "Current" else "Constant"
    print("━" * 70)
    print(f"  Full-year EEIO analysis ({label})")
    print("━" * 70)

    for yr, fpath in io_files.items():
        res = run_eeio(yr, fpath, ghg_df, p_type=p_type, exclude_keyword=exclude_keyword)
        if res is None:
            continue
        final_results[yr] = res
        res_sorted = res.sort_values("Supply chain induced coefficient (B*L)", ascending=False)
        csv_file   = out_path(f"Korea_EEIO_{p_type}_{yr}.csv", output_dir)
        res_sorted.to_csv(csv_file, encoding="utf-8-sig")
        print(f"    ✅ {yr} complete ({len(res_sorted)} sector) → {csv_file}")

    print("━" * 70)
    print(f"  analysis complete -- total {len(final_results)} Year")
    print("━" * 70)
    return final_results

# ═══════════════════════════════════════════════════════════════════════════════
# Consolidate data for visualization
# ═══════════════════════════════════════════════════════════════════════════════

def build_viz_data(final_results: dict):
    """
    final_results -> returns (total_viz, pivot_BL, pivot_B).

    Returns
    -------
    total_viz : long-format DataFrame
    pivot_BL  : supply chain induced coefficient pivot (Sector x Year)
    pivot_B   : direct emission coefficient pivot   (Sector x Year)
    """
    frames = []
    for yr, df in final_results.items():
        tmp = df.copy().reset_index()
        tmp["Year"] = int(yr)
        frames.append(tmp)

    total_viz = pd.concat(frames, ignore_index=True)

    pivot_BL = clean_and_merge_pivot(
        total_viz.pivot(index="Sector name", columns="Year", values="Supply chain induced coefficient (B*L)")
    )
    pivot_B = clean_and_merge_pivot(
        total_viz.pivot(index="Sector name", columns="Year", values="Direct emission coefficient (B)")
    )

    order_2023 = pivot_BL[2023].sort_values(ascending=False).index
    pivot_BL   = pivot_BL.loc[order_2023]
    pivot_B    = pivot_B.loc[order_2023]

    print(f"✅ Merge complete -- {len(total_viz['Sector name'].unique())} sectors x {len(total_viz['Year'].unique())} years")
    return total_viz, pivot_BL, pivot_B

# ═══════════════════════════════════════════════════════════════════════════════
# Scope 1/2/3 dashboard (donut + Upstream/Downstream cards)
# ═══════════════════════════════════════════════════════════════════════════════

def plot_dashboard_with_scope(
    final_results: dict,
    io_files: dict,
    p_type: str,
    year: str,
    exclude_keyword: str | None = None,
    output_dir: str = "./output",
) -> None:
    """
    Original current-price STEP 11 -- comprehensive Scope 1/2/3 dashboard per
    sector (EEIO-IDA style).
    Excludes only sector names that exactly match exclude_keyword (not
    startswith).

    Layout:
      1) Comprehensive intro cover page (overall insights)
      2) Sequential per-sector cards (donut + Upstream/Downstream + description)

    * Running this function once saves the intro/card PNGs and
      Dashboard_Index_{p_type}_{year}.json to output_dir, so afterwards you
      can re-display instantly without recomputation via
      show_dashboard_year().
    """
    import textwrap
    import matplotlib as mpl
    from matplotlib.patches import FancyBboxPatch

    mpl.rcParams['figure.dpi'] = 200

    # -- build df_scope -----------------------------------------------------------
    try:
        xls       = pd.ExcelFile(io_files[year])
        tot_sheet = _pick_sheet(xls, 'Total_Transaction', p_type)
        df_tot    = pd.read_excel(io_files[year], sheet_name=tot_sheet, header=None)
        s_d, e_d, _ = _find_block(df_tot)
        names_raw = df_tot.iloc[s_d:e_d + 1, 1].values
        vals_raw  = _total_input_vector(df_tot, len(names_raw))
        # The result of run_eeio() (b_dict etc.) uses names normalized by clean_industry_names(),
        # so we must normalize the same way here too, or the matching will break.
        # (e.g. if the 2020 original still has a notation with no space left over, like 'Broadcastingservices',
        #  it will differ from the normalized name and that sector will be silently skipped.)
        names_clean = clean_industry_names(pd.Index(names_raw.astype(str))).values
        univ_inp  = dict(zip(names_clean, vals_raw))
    except Exception as e:
        print(f"❌ load failed: {e} → uniform dummy value use")
        univ_inp = {k: 100.0 for k in final_results[year].index}

    res     = final_results[year]
    b_dict  = res['Direct emission coefficient (B)'].to_dict()
    s2_dict = res['Scope2'].to_dict()
    s3_dict = res['Scope3'].to_dict()
    bl_dict = res['Supply chain induced coefficient (B*L)'].to_dict()

    scope_rows = []
    for sec, val in univ_inp.items():
        sec_s = str(sec).strip()
        if exclude_keyword and sec_s == exclude_keyword:   # exclude only on an exact match
            continue
        if sec_s not in b_dict or val <= 0:
            continue
        scope_rows.append({
            'sector':     sec_s,
            'Input value': val,
            'Scope1':   val * b_dict[sec_s],
            'Scope2':   val * s2_dict[sec_s],
            'Scope3':   val * s3_dict[sec_s],
            'Total':     val * bl_dict[sec_s],
            'Carbon intensity': val * bl_dict[sec_s] / val,
        })

    df_scope = pd.DataFrame(scope_rows).sort_values('Total', ascending=False)

    # -- dashboard parameters ------------------------------------------------------
    DASH_YEAR     = year
    TOP_N_CONTRIB = 5
    MY_COLORS     = ['#1B4332', '#2D6A4F', '#74C69D']
    label         = 'Current' if p_type == 'Current' else 'Constant'

    df_dash   = df_scope.copy().reset_index(drop=True)
    n_sectors = len(df_dash)

    # ★ in the key p_type  including looks up (run_eeio()  savea the and identical rule).
    #   This way, even computing the same year alternately as Current/Constant does not overwrite
    #   the other, and uses the M/M_X that exactly corresponds to the p_type being drawn right now.
    _mkey = f"{DASH_YEAR}_{p_type}"
    if _mkey in M_matrices:
        mdat = M_matrices[_mkey]
    else:
        # If this exact key is not found (e.g. an older cache), fall back to the last computed result
        mdat = M_matrices.get(DASH_YEAR, list(M_matrices.values())[-1])
    sector_names_M = list(mdat['Sector name'])
    # -- Upstream/Downstream use the same values as the Excel M_X sheet
    #    (value-induced matrix, unit: t CO2eq). M_X = M(diag(B)*L, per-unit basis) x diag(X)(actual total input value).
    #    The kt -> t conversion (x1000) is already reflected inside run_eeio() when B is computed,
    #    so it is not multiplied again here (the previous version incorrectly multiplied by M*1000,
    #    which caused a mismatch with the Excel M_X value).
    M_X_mat        = mdat['M_X']

    def get_M_index(name):
        try:    return sector_names_M.index(name)
        except ValueError: return None

    def draw_sector_card(fig, gs_row, sector_row, rank):
        sec = sector_row['sector']
        scope1, scope2, scope3 = sector_row['Scope1'], sector_row['Scope2'], sector_row['Scope3']
        total_12  = scope1 + scope2
        total_123 = scope1 + scope2 + scope3
        if total_123 <= 0:
            total_123 = 1e-12

        idx = get_M_index(sec)
        if idx is not None:
            # col: idx sector(j)  when produced, each sector(i) "via"so that induced
            #      absolute emission amount(t CO2eq) → "Upstream(Backward)": idx the sector's
            #      to produce it -- i.e. which sectors' emissions are induced
            col      = M_X_mat[:, idx].copy(); col[idx] = 0
            up_idx   = np.argsort(col)[::-1][:TOP_N_CONTRIB]
            upstream_top = [(sector_names_M[i], col[i]) for i in up_idx]
            # rowv: how much sector i (idx)'s emissions are induced by each sector (j)'s production
            #      -- "Downstream (Forward)"
            rowv     = M_X_mat[idx, :].copy(); rowv[idx] = 0
            dn_idx   = np.argsort(rowv)[::-1][:TOP_N_CONTRIB]
            downstream_top = [(sector_names_M[i], rowv[i]) for i in dn_idx]
        else:
            upstream_top = downstream_top = []

        gs_card  = gs_row.subgridspec(1, 4, width_ratios=[1.15, 0.20, 0.9, 1.45], wspace=0.05)

        # left side panel
        ax_left = fig.add_subplot(gs_card[0, 0])
        ax_left.axis('off')
        ax_left.set_xlim(0, 1); ax_left.set_ylim(-0.6, 1.15)

        title_text    = f"#{rank}  {sec}"
        title_fontsize = 13 if len(title_text) <= 13 else (11.5 if len(title_text) <= 20 else 10.5)
        ax_left.text(0, 1.10, title_text, fontsize=title_fontsize, weight='bold', color='#16a34a', va='top')
        ax_left.text(0, 0.99, f"Korea EEIO analysis result | {DASH_YEAR} {label}", fontsize=8.5, color='#6b7280', va='top')

        box_y = 0.88
        ax_left.add_patch(FancyBboxPatch((0, box_y-0.085), 0.46, 0.11, boxstyle="round,pad=0.012",
                           facecolor='#f3f4f6', edgecolor='#d1d5db', transform=ax_left.transData, clip_on=False))
        ax_left.text(0.23, box_y-0.012, "Scope 1+2", fontsize=8.5, ha='center', color='#374151', weight='bold')
        ax_left.text(0.23, box_y-0.065, f"{total_12:,.4f}", fontsize=12, ha='center', color='#111827', weight='bold')

        ax_left.add_patch(FancyBboxPatch((0.50, box_y-0.085), 0.46, 0.11, boxstyle="round,pad=0.012",
                           facecolor='#dcfce7', edgecolor='#16a34a', transform=ax_left.transData, clip_on=False))
        ax_left.text(0.73, box_y-0.012, "Scope 1+2+3 (Total emissions)", fontsize=8.5, ha='center', color='#15803d', weight='bold')
        ax_left.text(0.73, box_y-0.065, f"{total_123:,.4f}", fontsize=12, ha='center', color='#15803d', weight='bold')

        ax_left.text(0, box_y-0.135, f"unit: t CO₂eq / KRW million ({label} basis)", fontsize=8, color='#9ca3af')

        line_gap = 0.085
        uy = box_y - 0.23
        ax_left.text(0, uy, "Top 5 Upstream  (other sectors' emissions actually induced by this sector's production, t CO2eq)", fontsize=9, weight='bold', color='#1B4332')
        for i, (nm, val) in enumerate(upstream_top or [("No data", 0)]):
            ax_left.text(0.02, uy - line_gap*(i+1), f"{i+1}. {nm}", fontsize=8.5, color='#374151')
            ax_left.text(0.98, uy - line_gap*(i+1), f"{val:,.4f}",   fontsize=8.5, color='#6b7280', ha='right')

        dy = uy - line_gap * 6.2
        ax_left.text(0, dy, "Top 5 Downstream  (amount actually induced in other sectors' production by this sector's emissions, t CO2eq)", fontsize=9, weight='bold', color='#2D6A4F')
        for i, (nm, val) in enumerate(downstream_top or [("No data", 0)]):
            ax_left.text(0.02, dy - line_gap*(i+1), f"{i+1}. {nm}", fontsize=8.5, color='#374151')
            ax_left.text(0.98, dy - line_gap*(i+1), f"{val:,.4f}",   fontsize=8.5, color='#6b7280', ha='right')

        # center panel: donut
        ax_donut = fig.add_subplot(gs_card[0, 2])
        vals = [max(scope1, 0), max(scope2, 0), max(scope3, 0)]
        if sum(vals) <= 0:
            vals = [1, 0, 0]
        wedges, _texts = ax_donut.pie(
            vals, colors=MY_COLORS, startangle=90,
            radius=0.85,
            wedgeprops={'width': 0.36, 'edgecolor': 'white', 'linewidth': 1.5}
        )
        # External callout: show the Scope name + % figure in the slice color
        _slabels = ['Scope 1\n(direct)', 'Scope 2\n(energy)', 'Scope 3\n(supply chain)']
        total_v  = sum(vals) or 1e-12
        for wedge, v, slbl in zip(wedges, vals, _slabels):
            pct = v / total_v * 100
            if pct < 0.01:
                continue  # only skip when it is effectively 0
            ang = (wedge.theta1 + wedge.theta2) / 2
            rad = np.deg2rad(ang)
            r_ring = 0.85 * (1 - 0.36 / 2)   # ring center radius
            r_tip  = 0.90                      # line start point
            r_bend = 1.05                      # the bend point
            r_text = 1.12                      # text
            _xt, _yt = r_text * np.cos(rad), r_text * np.sin(rad)
            _xb, _yb = r_bend * np.cos(rad), r_bend * np.sin(rad)
            _xs, _ys = r_tip  * np.cos(rad), r_tip  * np.sin(rad)
            # connector line (outer edge of the ring -> bend point)
            ax_donut.plot([_xs, _xb], [_ys, _yb],
                          color=wedge.get_facecolor(), lw=1.0, solid_capstyle='round')
            ha = 'left' if _xt >= 0 else 'right'
            ax_donut.text(
                _xt, _yt,
                f"{slbl}\n{pct:.1f}%",
                ha=ha, va='center', fontsize=7.5, weight='bold',
                color=wedge.get_facecolor(),
                linespacing=1.3,
            )
        ax_donut.set_xlim(-1.7, 1.7); ax_donut.set_ylim(-1.7, 1.7)
        sec_wrapped   = textwrap.fill(sec, width=10)
        name_fontsize = 12 if len(sec) <= 6 else (10.5 if len(sec) <= 12 else 9.5)
        ax_donut.text(0, 0, sec_wrapped, ha='center', va='center',
                      fontsize=name_fontsize, weight='bold', linespacing=1.25)

        # right side panel: Description
        ax_right = fig.add_subplot(gs_card[0, 3])
        ax_right.axis('off')
        ax_right.set_xlim(0, 1.25); ax_right.set_ylim(-0.4, 1.15)
        rx = 0.12
        ax_right.text(rx, 1.10, "Scope 1, 2, 3 emission structure", fontsize=11, weight='bold', color='#111827', va='top')
        sec_right_wrapped = textwrap.fill(sec, width=22)
        ax_right.text(rx, 0.98, sec_right_wrapped, fontsize=10, weight='bold', color='#16a34a', va='top', linespacing=1.3)

        n_right_lines = sec_right_wrapped.count('\n') + 1
        s1p = scope1 / total_123 * 100
        s2p = scope2 / total_123 * 100
        s3p = scope3 / total_123 * 100
        ty  = 0.82 - 0.08 * max(n_right_lines - 1, 0)
        gap_y = 0.30

        ax_right.text(rx, ty,      f"Scope 1 ({s1p:.0f}%)", fontsize=9.5, weight='bold', color=MY_COLORS[0])
        ax_right.text(rx, ty-0.08, f"Direct combustion and process emissions account for\n{s1p:.0f}% of this sector's emissions.",
                      fontsize=9, color='#374151', va='top', linespacing=1.4)
        ty2 = ty - gap_y
        ax_right.text(rx, ty2,      f"Scope 2 ({s2p:.0f}%)", fontsize=9.5, weight='bold', color=MY_COLORS[1])
        ax_right.text(rx, ty2-0.08, f"Indirect emissions arising from the use of purchased\nelectricity, gas, and steam are {s2p:.0f}%.",
                      fontsize=9, color='#374151', va='top', linespacing=1.4)
        ty3 = ty2 - gap_y
        ax_right.text(rx, ty3,      f"Scope 3 ({s3p:.0f}%)", fontsize=9.5, weight='bold', color=MY_COLORS[2])
        ax_right.text(rx, ty3-0.08, f"The remaining {s3p:.0f}% is induced by the supply\nchain as a whole (raw materials, transport, etc.).",
                      fontsize=9, color='#374151', va='top', linespacing=1.4)
        ax_right.text(rx, -0.30, f"* based on aggregated sector-level data approximation, {label} basis",
                      fontsize=7.5, color='#9ca3af', va='bottom')

    # -- 1. Intro cover page ---------------------------------------------------------
    total_all  = df_dash['Total'].sum() or 1e-12
    s1a, s2a, s3a = df_dash['Scope1'].sum(), df_dash['Scope2'].sum(), df_dash['Scope3'].sum()
    top3       = df_dash.head(3)
    top3_share = top3['Total'].sum() / total_all * 100
    s1_sh = (df_dash['Scope1'] / df_dash['Total'].replace(0, np.nan) * 100).fillna(0)
    s3_sh = (df_dash['Scope3'] / df_dash['Total'].replace(0, np.nan) * 100).fillna(0)

    insight_lines = [
        f"The combined carbon footprint of all {n_sectors} sectors is {total_all:,.4f} t CO2eq, "
        f"of which Scope 1 (direct) accounts for {s1a/total_all*100:.1f}%, Scope 2 (energy) for {s2a/total_all*100:.1f}%, "
        f"and Scope 3 (supply chain) for {s3a/total_all*100:.1f}%.",
        f"The top 3 sectors by total emissions ({', '.join(top3['sector'].tolist())}) account for "
        f"{top3_share:.1f}% of the total supply-chain carbon footprint, showing a structure where emissions are concentrated in a small number of key sectors.",
        f"The sector with the highest Scope 1 share is '{df_dash.loc[s1_sh.idxmax(),'sector']}' ({s1_sh.max():.0f}%), where on-site process and combustion emissions need urgent direct-emission management, and "
        f"the sector with the highest Scope 3 share is '{df_dash.loc[s3_sh.idxmax(),'sector']}' ({s3_sh.max():.0f}%), where supply-chain management is key to cutting carbon.",
        "Sectors with a higher Scope 3 share cannot rely on improving their own facilities alone, so a low-carbon transition strategy spanning raw-material sourcing and the transport/supply chain as a whole is needed."
    ]

    fig_intro = plt.figure(figsize=(22, 5.0), facecolor='white')
    ax_title  = fig_intro.add_axes([0, 0.7, 1, 0.3]); ax_title.axis('off')
    ax_title.text(0.5, 0.6, f"Korea, by sector, Scope 1, 2, 3 carbon emission comprehensive dashboard ({DASH_YEAR}, {label})",
                  fontsize=20, weight='bold', ha='center', va='center', color='#1B4332')
    ax_title.text(0.5, 0.1, f"{n_sectors} sectors total | sorted by total emissions, descending | Upstream/Downstream based on the carbon-induced (M matrix)",
                  fontsize=11, ha='center', va='center', color='#6b7280')

    ax_ins = fig_intro.add_axes([0.02, 0.0, 0.96, 0.65]); ax_ins.axis('off')
    ax_ins.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0.012",
                      facecolor='#f0fdf4', edgecolor='#16a34a', linewidth=1.3, transform=ax_ins.transAxes))
    ax_ins.text(0.02, 0.85, "Overall insights -- carbon emission structure across all sectors",
                fontsize=12, weight='bold', color='#15803d', va='top')
    iy = 0.65
    for line in insight_lines:
        block = "\n  ".join(textwrap.wrap(line, width=110))
        ax_ins.text(0.02, iy, f"- {block}", fontsize=10, color='#1f2937',
                    va='top', transform=ax_ins.transAxes, linespacing=1.5)
        iy -= 0.18

    from IPython.display import display as _display

    # -- 2. individual sector cards ------------------------------------------------------------
    # the filename includes (p_type, Year, Rank, Sector name) so the image alone can be reloaded later
    card_index = []
    for rank, (_, row) in enumerate(df_dash.iterrows(), start=1):
        fig_card = plt.figure(figsize=(22, 7.5), facecolor='white')
        draw_sector_card(fig_card, fig_card.add_gridspec(1, 1)[0], row, rank)
        safe = "".join(c for c in row['sector'] if c.isalnum() or c in " _-")
        card_fname = f'Dashboard_Card_{DASH_YEAR}_{rank:02d}_{safe}.png'
        fig_card.savefig(out_path(card_fname, output_dir),
                         dpi=200, bbox_inches='tight', facecolor='white')
        _display(fig_card)
        plt.close(fig_card)
        card_index.append({"rank": rank, "sector": row['sector'], "file": card_fname})

    # save the intro cover page + display on screen
    intro_fname = f'Dashboard_00_Intro_{DASH_YEAR}.png'
    fig_intro.savefig(out_path(intro_fname, output_dir),
                      dpi=300, bbox_inches='tight', facecolor='white')
    _display(fig_intro)
    plt.close(fig_intro)

    # save index file (year-to-sector-to-filename mapping; used by show_dashboard_year() etc, not by widgets)
    import json as _json
    idx_path = out_path(f'Dashboard_Index_{p_type}_{DASH_YEAR}.json', output_dir)
    with open(idx_path, 'w', encoding='utf-8') as f:
        _json.dump({"year": DASH_YEAR, "p_type": p_type,
                    "intro_file": intro_fname, "cards": card_index},
                   f, ensure_ascii=False, indent=2)

    print("✅ Every sector's dashboard card was generated/saved individually at high resolution.")

# ═══════════════════════════════════════════════════════════════════════════════
# Facet time-series chart (direct vs. supply-chain emissions, by sector)
# ═══════════════════════════════════════════════════════════════════════════════

def plot_facet_timeseries(
    total_viz, pivot_BL, analysis_years: list,
    p_type: str = "Constant",
    exclude_keyword: str | None = "Others",
    output_dir: str = "./output",
):
    """Constant-price STEP 9 facet time-series chart.
    Excludes only sectors exactly matching 'Others' and shows the time series for all sectors.
    """
    if exclude_keyword:
        sectors = [s for s in pivot_BL.index if str(s).strip() != exclude_keyword]
    else:
        sectors = list(pivot_BL.index)

    n_cols    = 4
    n_rows    = int(np.ceil(len(sectors) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.4), facecolor="white")
    axes      = axes.flatten()

    year_to_pos = {y: i for i, y in enumerate(analysis_years)}

    for i, sec in enumerate(sectors):
        ax  = axes[i]
        sub = total_viz[total_viz["Sector name"] == sec].sort_values("Year")
        x_pos = sub["Year"].map(year_to_pos)
        ax.plot(x_pos, sub["Direct emission coefficient (B)"],     marker="s", ms=5, lw=1.8, color="#2563eb", label="direct emissions (B)")
        ax.plot(x_pos, sub["Supply chain induced coefficient (B*L)"], marker="o", ms=5, lw=1.8, color="#16a34a", label="Total emissions (B x L)")
        ax.set_title(sec, fontsize=10, weight="bold")
        ax.set_xticks(range(len(analysis_years)))
        ax.set_xticklabels([str(y) for y in analysis_years])
        ax.tick_params(axis="x", rotation=45, labelsize=7.5)
        ax.tick_params(axis="y", labelsize=7.5)
        ax.grid(True, linestyle=":", alpha=0.4)
        if i == 0:
            ax.legend(fontsize=7.5, loc="best")
    for j in range(len(sectors), len(axes)):
        axes[j].axis("off")

    fig.suptitle(
        f"by sector Direct emission coefficient (B) vs total emission coefficient (B x L) time-series comparison\n"
        f"({p_type} prices, all sectors, unit: t CO₂eq / KRW million)",
        fontsize=15, weight="bold", y=1.02,
    )
    plt.tight_layout()
    fpath = out_path(f"Step_B_vs_BL_Timeseries_{p_type}.png", output_dir)
    plt.savefig(fpath, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"🖼️  Saved -> {fpath}")


# ── file / analysis configuration ──────────────────────────────────────────
OUTPUT_DIR = './output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IO_FILES = {
    '2015': 'O_2015_IO_Table_Current_Constant_Final_EN.xlsx',
    '2020': 'O_2020_IO_Table_Current_Constant_Final_EN.xlsx',
    '2021': 'O_2021_IO_Table_Current_Constant_Final_EN.xlsx',
    '2022': 'O_2022_IO_Table_Current_Constant_Final_EN.xlsx',
    '2023': 'O_2023_IO_Table_Current_Constant_Final_EN.xlsx',
}
ANALYSIS_YEARS = [2015, 2020, 2021, 2022, 2023]
GHG_FILE = 'O_2015-2023_Industry_GHG_Estimates_FF_EN.xlsx'
EXCLUDE  = 'Others'
print('✅ Setup complete (running directly on the eeio_core code inlined above -- no eeio_core import needed)')


---

# 📗 Current-Price Analysis



## 📋 GHG Emissions Data Preview

Greenhouse gas emissions for Korea's 33 sectors, 2015-2023.
The source unit is kt CO2eq; it is shown here converted with **x1000 -> t CO2eq**.


In [ ]:
clean_ghg_c = load_ghg_data(GHG_FILE)

_year_cols = [c for c in clean_ghg_c.columns if c.isdigit() and 2000 <= int(c) <= 2100]
_preview = clean_ghg_c[['Code', 'Sector name'] + _year_cols].copy()
_preview[_year_cols] = (_preview[_year_cols] * 1000).round(1)

display(HTML(
    "<div style='background:#f0fdf4;padding:10px;border-left:5px solid #16a34a;margin-bottom:10px'>"
    "<b>📋 GHG data preview (unit: t CO2eq)</b></div>"
))
display(_preview)


---

# 📗 Current-Price Analysis



## 🧭 Comprehensive Scope 1/2/3 Dashboard, by Sector (Current Prices)

Computes EEIO coefficients for all years directly, then generates and displays the intro
summary cover page and every sector's card, in order, for the chosen year.


In [ ]:
# Change the year and run (e.g. '2015', '2020', '2021', '2022', '2023')
_year = '2023'

final_results_c = run_all_years(
    IO_FILES, ANALYSIS_YEARS, clean_ghg_c,
    p_type='Current', exclude_keyword=EXCLUDE, output_dir=OUTPUT_DIR,
)

plot_dashboard_with_scope(
    final_results_c, IO_FILES, p_type='Current',
    year=_year, exclude_keyword=EXCLUDE, output_dir=OUTPUT_DIR,
)


---

# 📊 EEIO Matrix Excel Files

The analysis-result Excel files are already saved in the `output/` folder, by year and price basis.

| Sheet | Content |
|------|------|
| **Description** | Symbol/description for each sheet |
| **T_mat_GHG** | Total Transaction table + GHG emissions row |
| **Ad** | Domestic input coefficients |
| **Lf** | Leontief inverse matrix |
| **M** | GHG emission-induced coefficient diag(B)*Lf |
| **M.Fd** | Emissions by final-demand component |
| **M.X(diag)** | Value-induced matrix (M x diag(X)) |
| **Scopes** | Scope 1/2/3 (correction for double-counting in the power sector) |
| **I_mat** | Import Transaction table |
| **D_mat** | Domestic Transaction table |



## 📁 Save Location

Files are saved by year and price basis in the form `output/{year}_KR_EEIO_{Current|Constant}.xlsx`.
e.g. `output/2023_KR_EEIO_Current.xlsx`, `output/2023_KR_EEIO_Constant.xlsx`
(generated by the separate Excel-calculation notebook, `eeio_core.py`'s `save_eeio_excel()`).



In [ ]:
import glob

_xlsx_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*_KR_EEIO_*.xlsx')))
print(f'📁 EEIO Excel files saved in {OUTPUT_DIR} ({len(_xlsx_files)}):')
for f in _xlsx_files:
    print(' -', os.path.basename(f))


---

# 📘 Current-Price Analysis



## 📋 Direct Emission Coefficient (B) vs. Total Emission Coefficient (B x L) Time Series, by Sector (Current Prices)

In [ ]:
viz_c, pivot_BL_c, pivot_B_c = build_viz_data(final_results_c)

plot_facet_timeseries(
    viz_c, pivot_BL_c, ANALYSIS_YEARS,
    p_type='Current', exclude_keyword=EXCLUDE, output_dir=OUTPUT_DIR,
)


---

# 📘 Constant-Price Analysis



## 📋 Direct Emission Coefficient (B) vs. Total Emission Coefficient (B x L) Time Series, by Sector (Constant Prices)



In [ ]:
clean_ghg_k = load_ghg_data(GHG_FILE)

final_results_k = run_all_years(
    IO_FILES, ANALYSIS_YEARS, clean_ghg_k,
    p_type='Constant', exclude_keyword=EXCLUDE, output_dir=OUTPUT_DIR,
)

viz_k, pivot_BL_k, pivot_B_k = build_viz_data(final_results_k)

plot_facet_timeseries(
    viz_k, pivot_BL_k, ANALYSIS_YEARS,
    p_type='Constant', exclude_keyword=EXCLUDE, output_dir=OUTPUT_DIR,
)
